In [0]:
%%sql
CREATE VOLUME IF NOT EXISTS workspace.default.datalake;
--/Volumes/workspace/default/datalake
-- carregar dados

In [0]:
import os

directory = "/Volumes/workspace/default/datalake/"
if os.path.exists(directory):
    files = os.listdir(directory)
    for file in files:
        if file.split(".")[-1] == "csv":
            filepath = os.path.join(directory, file)
            filesulfix = file.split(".")[0]
            try:
                df = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load(filepath)
                df.write.format("delta").save("/Volumes/workspace/default/datalake/delta/" + filesulfix + ".delta")
            except Exception as e:
                print(f"Error processing file {file}: {e}")
else:
    print("The directory does not exist.")

In [0]:
! ls /Volumes/workspace/default/datalake/delta

categories.delta  employees.delta     orders.delta    shippers.delta
customers.delta   orderdetails.delta  products.delta  suppliers.delta


In [0]:
"""
A celula anterior com um for loop substitui essa celula

#criar delta a partir do csv
categories = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/categories.csv")
categories.write.format("delta").save("/Volumes/workspace/default/datalake/delta/categories.delta")

customers = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/customers.csv")
customers.write.format("delta").save("/Volumes/workspace/default/datalake/delta/customers.delta")

employees = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/employees.csv")
employees.write.format("delta").save("/Volumes/workspace/default/datalake/delta/employees.delta")

orderdetails = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/orderdetails.csv")
orderdetails.write.format("delta").save("/Volumes/workspace/default/datalake/delta/orderdetails.delta")

products = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/products.csv")
products.write.format("delta").save("/Volumes/workspace/default/datalake/delta/products.delta")

shippers = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/shippers.csv")
shippers.write.format("delta").save("/Volumes/workspace/default/datalake/delta/shippers.delta")

suppliers = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/suppliers.csv")
suppliers.write.format("delta").save("/Volumes/workspace/default/datalake/delta/suppliers.delta")

orders = spark.read.format("csv").option("header", "true").option("delimiter", ";").option("inferSchema", "true").load("/Volumes/workspace/default/datalake/orders.csv")
orders.write.format("delta").save("/Volumes/workspace/default/datalake/delta/orders.delta")
"""

In [0]:
#ler delta
products = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/products.delta")
products.show()

+---------+--------------------+----------+----------+--------------------+---------+------------+------------+------------+------------+
|ProductID|         ProductName|SupplierID|CategoryID|     QuantityPerUnit|UnitPrice|UnitsInStock|UnitsOnOrder|ReorderLevel|Discontinued|
+---------+--------------------+----------+----------+--------------------+---------+------------+------------+------------+------------+
|        1|                Chai|         1|         1|  10 boxes x 20 bags|     18.0|          39|           0|          10|           0|
|        2|               Chang|         1|         1|  24 - 12 oz bottles|     19.0|          17|          40|          25|           0|
|        3|       Aniseed Syrup|         1|         2| 12 - 550 ml bottles|     10.0|          13|          70|          25|           0|
|        4|Chef Anton's Caju...|         2|         2|      48 - 6 oz jars|     22.0|          53|           0|           0|           0|
|        5|Chef Anton's Gumb...|  

In [0]:
#ler delta
categories = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/categories.delta")
print(categories.schema)
categories.show(truncate=False)


StructType([StructField('CategoryID', IntegerType(), True), StructField('CategoryName', StringType(), True), StructField('Description', StringType(), True)])
+----------+--------------+----------------------------------------------------------+
|CategoryID|CategoryName  |Description                                               |
+----------+--------------+----------------------------------------------------------+
|1         |Beverages     |Soft drinks, coffees, teas, beers, and ales               |
|2         |Condiments    |Sweet and savory sauces, relishes, spreads, and seasonings|
|3         |Confections   |Desserts, candies, and sweet breads                       |
|4         |Dairy Products|Cheeses                                                   |
|5         |Grains/Cereals|Breads, crackers, pasta, and cereal                       |
|6         |Meat/Poultry  |Prepared meats                                            |
|7         |Produce       |Dried fruit and bean curd       

In [0]:
#inserir no delta
df = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/categories.delta")

novacategoria = spark.createDataFrame([(9, "coffee", "Moka pot, Aeropress, cappuccino")], df.schema)

novacategoria.write.format("delta").mode("append").save("/Volumes/workspace/default/datalake/delta/categories.delta")

In [0]:
categories = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/categories.delta")
categories.show()

+----------+--------------+--------------------+
|CategoryID|  CategoryName|         Description|
+----------+--------------+--------------------+
|         1|     Beverages|Soft drinks, coff...|
|         2|    Condiments|Sweet and savory ...|
|         3|   Confections|Desserts, candies...|
|         4|Dairy Products|             Cheeses|
|         5|Grains/Cereals|Breads, crackers,...|
|         6|  Meat/Poultry|      Prepared meats|
|         7|       Produce|Dried fruit and b...|
|         8|       Seafood|    Seaweed and fish|
|         9|        coffee|Moka pot, Aeropre...|
+----------+--------------+--------------------+



In [0]:
orders = spark.read.format("delta") \
    .load("/Volumes/workspace/default/datalake/delta/orders.delta") \
    .filter("OrderID = 11078") \
    .select("OrderID")
orders.show()

orderdetails = spark.read.format("delta") \
    .load("/Volumes/workspace/default/datalake/delta/orderdetails.delta") \
    .filter("OrderID = 11078") \
    .select("OrderID")
orderdetails.show()



+-------+
|OrderID|
+-------+
+-------+

+-------+
|OrderID|
+-------+
+-------+



In [0]:
#upsert 
from delta.tables import *

# Carregar tabelas Delta como DeltaTable
deltaTable_orders = DeltaTable.forPath(spark, "/Volumes/workspace/default/datalake/delta/orders.delta")
deltaTable_order_details = DeltaTable.forPath(spark, "/Volumes/workspace/default/datalake/delta/orderdetails.delta")

# Criar os novos registros que queremos inserir
new_order = spark.createDataFrame([(11078, "ALFKI", 1, "2023-08-01")], ["OrderID", "CustomerID", "EmployeeID", "OrderDate"])
new_order_details = spark.createDataFrame([(11078, 1, 18, 3)], ["OrderID", "ProductID", "UnitPrice", "Quantity"])

deltaTable_orders.alias("orders").merge(
    new_order.alias("newOrder"),
    "orders.OrderID = newOrder.OrderID")\
    .whenMatchedUpdate(set = {"CustomerID" : "newOrder.CustomerID", "EmployeeID" : "newOrder.EmployeeID", "OrderDate" : "newOrder.OrderDate"})\
    .whenNotMatchedInsert(values = {"OrderID" : "newOrder.OrderID", "CustomerID" : "newOrder.CustomerID", "EmployeeID" : "newOrder.EmployeeID", "OrderDate" : "newOrder.OrderDate"})\
    .execute()

deltaTable_order_details.alias("order_details").merge(
    new_order_details.alias("newOrderDetails"),
    "order_details.OrderID = newOrderDetails.OrderID AND order_details.ProductID = newOrderDetails.ProductID")\
    .whenMatchedUpdate(set = {"UnitPrice" : "newOrderDetails.UnitPrice", "Quantity" : "newOrderDetails.Quantity"})\
    .whenNotMatchedInsert(values = {"OrderID" : "newOrderDetails.OrderID", "ProductID" : "newOrderDetails.ProductID", "UnitPrice" : "newOrderDetails.UnitPrice", "Quantity" : "newOrderDetails.Quantity"})\
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Ler a tabela Delta e filtrar por uma determinada condição
orders = spark.read.format("delta") \
    .load("/Volumes/workspace/default/datalake/delta/orders.delta") \
    .filter("OrderID = 11078")
orders.show()

orderdetails = spark.read.format("delta") \
    .load("/Volumes/workspace/default/datalake/delta/orderdetails.delta") \
    .filter("OrderID = 11078")
orderdetails.show()


+-------+----------+----------+-------------------+------------+-----------+-------+-------+--------+-----------+--------+----------+--------------+-----------+
|OrderID|CustomerID|EmployeeID|          OrderDate|RequiredDate|ShippedDate|ShipVia|Freight|ShipName|ShipAddress|ShipCity|ShipRegion|ShipPostalCode|ShipCountry|
+-------+----------+----------+-------------------+------------+-----------+-------+-------+--------+-----------+--------+----------+--------------+-----------+
|  11078|     ALFKI|         1|2023-08-01 00:00:00|        NULL|       NULL|   NULL|   NULL|    NULL|       NULL|    NULL|      NULL|          NULL|       NULL|
+-------+----------+----------+-------------------+------------+-----------+-------+-------+--------+-----------+--------+----------+--------------+-----------+

+-------+---------+---------+--------+--------+
|OrderID|ProductID|UnitPrice|Quantity|Discount|
+-------+---------+---------+--------+--------+
|  11078|        1|     18.0|       3|    NULL|
+-

In [0]:
#criar tabela com finalidade especifica

# Carregar as tabelas Delta em DataFrames do Spark
df_categories = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/categories.delta")
df_categories.createOrReplaceTempView("categories")

df_products = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/products.delta")
df_products.createOrReplaceTempView("products")

df_suppliers = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/suppliers.delta")
df_suppliers.createOrReplaceTempView("suppliers")

df_employees = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/employees.delta")
df_employees.createOrReplaceTempView("employees")

df_order_details = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/orderdetails.delta")
df_order_details.createOrReplaceTempView("order_details")

df_orders = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/orders.delta")
df_orders.createOrReplaceTempView("orders")

df_shippers = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/shippers.delta")
df_shippers.createOrReplaceTempView("shippers")

df_customers = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/customers.delta")
df_customers.createOrReplaceTempView("customers")

join_query = """
SELECT order_details.OrderID AS OrderID, order_details.Quantity , order_details.UnitPrice as UnitPrice,
products.ProductID as ProductID,  products.ProductName as Product, suppliers.CompanyName AS Suppliers,  
employees.LastName as Employee, orders.OrderDate as Date, customers.CompanyName as Customer
FROM orders
JOIN order_details ON orders.OrderID = order_details.OrderID
JOIN products ON order_details.ProductID = products.ProductID
JOIN categories ON products.CategoryID = categories.CategoryID
JOIN suppliers ON products.SupplierID = suppliers.SupplierID
JOIN employees ON orders.EmployeeID = employees.EmployeeID
JOIN shippers ON orders.ShipVia = shippers.ShipperID
JOIN customers ON orders.CustomerID = customers.CustomerID
"""

df_result = spark.sql(join_query)

# Escrever o resultado em uma nova tabela Delta
df_result.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/datalake/delta/join.delta")

In [0]:
#consultar a tabela no delta com SQL
df = spark.read.format("delta").load("/Volumes/workspace/default/datalake/delta/join.delta")

df.createOrReplaceTempView("OrdersJoin")

results = spark.sql("SELECT * FROM OrdersJoin WHERE OrderID = 10248 AND ProductID =11 ")

results.show()

+-------+--------+---------+---------+--------------+--------------------+--------+-------------------+--------------------+
|OrderID|Quantity|UnitPrice|ProductID|       Product|           Suppliers|Employee|               Date|            Customer|
+-------+--------+---------+---------+--------------+--------------------+--------+-------------------+--------------------+
|  10248|      12|     14.0|       11|Queso Cabrales|Cooperativa de Qu...|Buchanan|2020-07-04 00:00:00|Vins et alcools C...|
+-------+--------+---------+---------+--------------+--------------------+--------+-------------------+--------------------+



In [0]:
dbutils.fs.rm("/Volumes/workspace/default/datalake/", recurse=True)